# 03 · Constraints

`fix=` holds a value exactly and `constrain=` keeps it inside a window, with 0-based atom indices as keys: two for
a distance in angstrom, three for an angle and four for a dihedral in degrees. `.measure()` reads each back.

In [ ]:
import numpy as np
import rxembed as rx
import xyzrender
from rdkit import Chem

## Name atoms with rx.match

`rx.match` returns the atom indices of the one SMARTS match and raises for none or several. `rx.embed` adds
hydrogens after the heavy atoms in RDKit's `AddHs` order, so matching on `Chem.AddHs` names a hydrogen too.

In [ ]:
PAIR = "CC(=O)O.c1ccncc1"  # acetic acid and pyridine
oxygen, hydrogen, nitrogen = rx.match(Chem.AddHs(rx.parse_smiles(PAIR)), "[OX2]-[H].n")
print(oxygen, hydrogen, nitrogen)

## Windows

A window is a `(low, high)` pair and a single number stands for a narrow window around it; here an H...N distance
and an O-H...N angle hold the acid's O-H on the pyridine N. Nothing pulls a value toward the middle of its window,
so it often ends at an edge.

In [ ]:
windows = {(hydrogen, nitrogen): (1.7, 1.9), (oxygen, hydrogen, nitrogen): (165, 180)}
pair = rx.embed(PAIR, constrain=windows, n=4, seed=1).minimize()
print(f"H...N {pair.measure((hydrogen, nitrogen))['mean']:.2f} A")
print(f"O-H...N {pair.measure((oxygen, hydrogen, nitrogen))['mean']:.1f} deg")
xyzrender.render(xyzrender.load(pair.lowest(1).mol, nci_detect=True))

## Fixed values

Butanol keeps a fixed gauche C-C-C-C dihedral and C-C-C angle exactly, while a window keeps its C-C-C-O dihedral
anti. Dihedral windows are periodic: `(170, 190)` runs through 180 to -170 degrees.

In [ ]:
butanol = rx.embed(
    "CCCCO", fix={(0, 1, 2, 3): 60.0, (1, 2, 3): 118.0}, constrain={(1, 2, 3, 4): (170, 190)}, n=4, seed=1
).minimize()
for key in [(0, 1, 2, 3), (1, 2, 3), (1, 2, 3, 4)]:
    print(key, round(butanol.measure(key)["mean"], 2))
xyzrender.render(xyzrender.load(butanol.lowest(1).mol))

## A pi stack

Two rings as a key hold them parallel at a centroid separation, atom for atom in the order given. Here the two
phenyls of 1,4-diphenylbutane stack at 3.7 angstrom.

In [ ]:
DIPHENYL = "c1ccccc1CCCCc1ccccc1"
ring_a, ring_b = Chem.AddHs(rx.parse_smiles(DIPHENYL)).GetSubstructMatches(Chem.MolFromSmarts("c1ccccc1"))
stack = rx.embed(DIPHENYL, constrain={(ring_a, ring_b): 3.7}, n=4, seed=1).minimize()
xyz = stack.mol.GetConformer(stack.ids[0]).GetPositions()
print(f"centroid separation {np.linalg.norm(xyz[list(ring_a)].mean(0) - xyz[list(ring_b)].mean(0)):.2f} A")
xyzrender.render(xyzrender.load(stack.lowest(1).mol))

## A rigid core

A list of atoms in `fix=` keeps their geometry from the input conformer exactly, while the core moves as a whole
and everything else is sampled. Here the acid and its H-bond to the pyridine N stay fixed and the ring turns about
that N.

In [ ]:
MOTIF = "[CX4][CX3](=O)[OX2][H].[nX2]"  # acid, its O-H and the pyridine N
reference = pair.lowest(1)
core = rx.match(reference.mol, MOTIF)
held = rx.embed(reference.mol, fix=list(core), n=4, seed=1).minimize()
print(held)
xyzrender.render(xyzrender.load(held.mol, ensemble=True))

## A template onto another molecule

`rx.Template(reference, smarts)` copies the reference coordinates of the matched atoms onto the same match in a new
molecule. Here the acetic acid and pyridine H-bond carries over to ibuprofen and DMAP.

In [ ]:
grafted = rx.embed(
    "CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O.CN(C)c1ccncc1", fix=rx.Template(reference, MOTIF), n=4, seed=1
).minimize()
*_, h, n = rx.match(grafted.mol, MOTIF)
print(grafted)
print(f"H...N {grafted.measure((h, n))['mean']:.2f} A, as in the reference")
xyzrender.render(xyzrender.load(grafted.lowest(1).mol, nci_detect=True))

## An impossible request

A request no geometry can meet raises and says which argument to loosen. Here a C-C bond is asked to be 5 angstrom
long.

In [ ]:
try:
    rx.embed("CCO", fix={(0, 1): 5.0}, n=2)
except ValueError as error:
    print(error)